# M-T — Translation moduli: original va tarjima qilingan matnni solishtirish

**Maqsad:** `data/raw/04_mixed_language.pdf` (o'zbekcha + ruscha + inglizcha) ustida
o'zbekcha -> inglizcha tarjimasini bajarib, natijani qo'lda tekshirish.

Bu notebook **o'rganish** uchun. Rasmiy test emas — testlar `tests/test_translation.py` da.

## 1. Konsept
Tarjima — matnni bir tildan ikkinchisiga o'tkazish. Bu yerda biz **chunk level emas,
page level** tarjima qilamiz (ADR-003). Sababi: kontekst. Bitta sahifa o'z ichida
izchil kontekstga ega, bitta chunk esa yo'q.

## 2. Intuition
Nega tarjima umuman kerak? Chunki keyingi bosqichlar (TF-IDF, K-Means, embedding)
ingliz tiliga yaxshiroq moslashgan. Ingliz matn bilan `stop_words='english'` ishlaydi,
ko'p tilli matn bilan esa `stop_words=None` kerak.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


import pymupdf

from app.translation import (
    GoogleTranslateTranslator,
    NoopTranslator,
    get_translator,
    validate_batch_mode,
)
from app.core.config import Settings

PDF = ROOT / "data" / "raw" / "04_mixed_language.pdf"
PDF.exists() or print("DIQQAT: PDF topilmadi — avval `python scripts/make_sample_pdfs.py` ishga tushiring")

doc = pymupdf.open(PDF)
pages = [p.get_text() for p in doc]
print(f"{PDF.name}: {len(pages)} sahifa, {sum(len(p) for p in pages)} belgi")

04_mixed_language.pdf: 6 sahifa, 3042 belgi


## 3. Matndagi tillar

Avval tildan tasdiqlaymiz — Kirill harflari mavjudmi?

In [2]:
full_text = "".join(pages)
cyrillic = [c for c in full_text if "\u0410" <= c <= "\u044f"]
latin = [c for c in full_text if c.isascii() and c.isalpha()]

print(f"Kirill harflari: {len(cyrillic)} ({len(set(cyrillic))} xil)")
print(f"Lotin harflari:  {len(latin)}")
print(f"'?' belgisi:      {full_text.count('?')}  (0 bo'lishi kerak)")

Kirill harflari: 561 (37 xil)
Lotin harflari:  1723
'?' belgisi:      0  (0 bo'lishi kerak)


> **M0 da topilgan nuqson:** `make_sample_pdfs.py` `fontname="helv"` (Helvetica)
> ishlatardi. Helvetica **faqat lotin** harflarni qo'llaydi — ruscha matn `??????`
> bo'lib yozilardi va tarjima moduli uni tarjima qila olmasdi. Endi `fontfile` bilan
> DejaVuSans yuklanadi.

## 4. Default: NoopTranslator (tashqi so'rov yo'q)

Avval default holatni tekshiramiz — `TRANSLATION_ENABLED=false`.

Muhim: bu holatda **tarmoqqa umuman so'rov yuborilmaydi** (test: `test_noop_makes_no_network_call`).

In [3]:
noop = get_translator(Settings(translation_enabled=False))
print(type(noop).__name__, "| enabled:", noop.enabled, "| healthcheck:", noop.healthcheck())

noop_out = noop.translate_pages(pages, "en")
print("Identik:", noop_out == pages)

NoopTranslator | enabled: False | healthcheck: True
Identik: True


## 5. Chunk level taqiqlangan (ADR-003)

Eng muhim qoida. Chunk level tarjima kontekstni buzadi, shuning uchun **kod darajasida**
taqiqlangan — config'ga ham, konstruktorga ham.

In [4]:
for mode in ("document", "page", "chunk"):
    try:
        print(f"  {mode:9} -> {validate_batch_mode(mode)}")
    except Exception as exc:
        print(f"  {mode:9} -> {type(exc).__name__}: {str(exc)[:70]}")

try:
    NoopTranslator(batch_mode="chunk")
except Exception as exc:
    print(f"\nNoopTranslator(chunk) -> {type(exc).__name__} (konstruktor ham bloklaydi)")

  document  -> document
  page      -> page
  chunk     -> TranslationConfigError: Taqiqlangan tarjima darajasi: 'chunk'. Ruxsat etilgan: document | page

NoopTranslator(chunk) -> TranslationConfigError (konstruktor ham bloklaydi)


## 6. Haqiqiy tarjima (o'zbekcha + ruscha -> inglizcha)

Endi `GoogleTranslateTranslator`. Bu **tashqi servis** — internet kerak.

> Xavfsizlik (security §55): har uchinchi servis uchun baholanadi —
> *qanaqa ma'lumot yuboriladi*, *maxfiylik*, *xarajat*, *ishlamasa nima bo'ladi*.
> Bu yerda yuboriladigan narsa — **foydalanuvchining PDF matni**. Bu maxfiylik
> muammosi bo'lishi mumkin, shuning uchun default **OCHIQ** qilingan.

In [5]:
translator = get_translator(
    Settings(
        translation_enabled=True,
        translation_provider="google_translate",
        google_translate_timeout=25.0,
        google_translate_max_retries=2,
    )
)
print(type(translator).__name__, "| enabled:", translator.enabled)
print("healthcheck:", translator.healthcheck())

GoogleTranslateTranslator | enabled: True


healthcheck: True


In [6]:
import time

start = time.perf_counter()
translated = translator.translate_pages(pages, "en")
elapsed = time.perf_counter() - start

print(f"{len(translated)} sahifa tarjima qilindi — {elapsed:.1f} soniya ({elapsed/len(translated):.1f} s/sahifa)")
print("Tartib va uzunlik saqlanadimi:", len(translated) == len(pages))

6 sahifa tarjima qilindi — 0.8 soniya (0.1 s/sahifa)
Tartib va uzunlik saqlanadimi: True


## 7. Qo'lda tekshirish

Natijani **har bir sahifa uchun** ko'rib chiqamiz. Noto'g'ri tarjima bo'lsa,
keyingi bosqichlarda klastr chalkashadi — shuning uchun bu qadam majburiy.

In [7]:
def show(text: str, width: int = 88) -> str:
    return " ".join(text.split())[:width]


for i, (src, out) in enumerate(zip(pages, translated), start=1):
    print(f"--- Sahifa {i} ({len(src)} -> {len(out)} belgi) ---")
    print("  asl :", show(src))
    print("  tarj:", show(out))
    print()

--- Sahifa 1 (572 -> 577 belgi) ---
  asl : Chapter 1. Sun'iy intellekt va machine learning ========================================
  tarj: Chapter 1. Artificial intelligence and machine learning ================================

--- Sahifa 2 (526 -> 501 belgi) ---
  asl : Chapter 2. Ma'lumotlarni tahlil qilish (Data Science) ==================================
  tarj: Chapter 2. Data analysis (Data Science) ================================================

--- Sahifa 3 (448 -> 441 belgi) ---
  asl : Chapter 3. Maшинное обучение (Russian) ======================================= Машинное 
  tarj: Chapter 3. Machine learning (English) ==================================================

--- Sahifa 4 (446 -> 474 belgi) ---
  asl : Chapter 4. Нейронные сети и deep learning ========================================== Ней
  tarj: Chapter 4. Neural networks and deep learning ===========================================

--- Sahifa 5 (508 -> 487 belgi) ---
  asl : Chapter 5. Tashqi arxitektura va

## 8. O'lchovlar: tarjima nimani o'zgartiradi?

M5 va M9 uchun muhim: tarjima matn uzunligini va token sonini o'zgartiradi.
Bu chunking parametrlarini (`chunk_size=800`) qayta ko'rib chiqishni talab qilishi mumkin.

In [8]:
print(f"{'Sahifa':>8} {'Asl':>7} {'Tarjima':>8} {"O'garish":>9}")
print("-" * 38)

ratios = []
for i, (src, out) in enumerate(zip(pages, translated), start=1):
    ratio = len(out) / len(src)
    ratios.append(ratio)
    print(f"{i:>8} {len(src):>7} {len(out):>8} {ratio:>8.2f}x")

print("-" * 38)
print(f"O'rtacha kengayish: {sum(ratios)/len(ratios):.2f}x")

  Sahifa     Asl  Tarjima  O'garish
--------------------------------------
       1     572      577     1.01x
       2     526      501     0.95x
       3     448      441     0.98x
       4     446      474     1.06x
       5     508      487     0.96x
       6     542      540     1.00x
--------------------------------------
O'rtacha kengayish: 0.99x


## 9. Texnik atomalar saqlanadimi?

Bu eng muhim savol. Agar tarjimada `machine learning` -> `mashinali o'rganish`
bo'lgan bo'lsa, TF-IDF texnik atamalarni yo'qotadi va klastr sifati pasayadi.

In [9]:
terms = [
    "machine learning", "neural network", "data science", "cloud",
    "deep learning", "classification", "regression", "embedding",
]

combined = " ".join(translated).lower()
print(f"{'Atama':22} {'Saqlanganmi':>13}")
print("-" * 37)
for term in terms:
    print(f"{term:22} {str(term in combined):>13}")

Atama                    Saqlanganmi
-------------------------------------
machine learning                True
neural network                  True
data science                    True
cloud                           True
deep learning                   True
classification                  True
regression                      True
embedding                      False


## 10. Xulosa — qanday qaror qabul qilaman?

Yuqoridagi natijalarga qarab quyidagi savolga javob berish kerak:

> **Aralash-tilli PDF uchun multilingual embedding (M9) yoki tarjima (M-T) yaxshimi?**

Mezonlar (M17 uchun ham xuddi shular):

| Mezon | Multilingual | Tarjima |
|-------|--------------|---------|
| Maxfiylik | ✅ matn chiqmaydi | ❌ tashqi serviska yuboriladi |
| Offline | ✅ ishlaydi | ❌ internet kerak |
| Xarajat | ✅ bepul | ✅ bepul (rate limit bilan) |
| Texnik atomalar | ❌ ko'p tilli model ham chalkashadi | ✅ saqlanadi (yuqoriga qarang) |
| Qishloq ishi (chiqish vaqti) | ✅ 1 so'rov | ❌ N so'rov (sahifa uchun) |
| Barqarorlik | ✅ | ❌ tashqi servisga bog'liq |

**Dastlabki qaror:** default `TRANSLATION_ENABLED=false` + multilingual embedding.
Tarjima — **zaxira yo'l**: multilingual natija sifatsiz bo'lsa, M17 da yoqiladi.

> Bu qaror **M17** da haqiqiy baholash bilan tasdiqlanadi. Hozircha bu —
> **gipoteza**, o'lchov emas.

## 11. Xotirjamlik (resources yopish)

In [10]:
translator._close()
print("HTTP klient yopildi.")

HTTP klient yopildi.
